# Data Analysis ENGIE 2026 — House Price Prediction\n**Built with Shimu**\n\nExplore the data, engineer features, compare models using hold-out MAPE, then create a five-seed CatBoost submission. In Kaggle, click **Add Input** and add the competition dataset.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from sklearn.ensemble import ExtraTreesRegressor
from catboost import CatBoostRegressor

# Kaggle normally mounts competition files under /kaggle/input/
ROOT = Path("/kaggle/input")
train_files, test_files = list(ROOT.rglob("train.csv")), list(ROOT.rglob("test.csv"))
print("train candidates:", train_files[:5])
print("test candidates:", test_files[:5])
if not train_files or not test_files:
    raise FileNotFoundError("Add the competition dataset in Kaggle > Add Input, then check ROOT.")
train, test = pd.read_csv(train_files[0]), pd.read_csv(test_files[0])
print("Train:", train.shape, "Test:", test.shape)
display(train.head())


## 1. EDA: target, missing values and correlations

In [ ]:
display(train.describe(include="all").T.head(30))
display(train["SalePrice"].describe())
missing = (train.isna().mean()*100).sort_values(ascending=False)
display(missing[missing > 0].head(20).rename("missing_percent").to_frame())
plt.figure(figsize=(8,4)); plt.hist(train["SalePrice"], bins=30)
plt.title("SalePrice distribution"); plt.xlabel("SalePrice"); plt.ylabel("Count"); plt.show()
numeric = train.select_dtypes(include=np.number)
if "SalePrice" in numeric:
    corr = numeric.corr(numeric_only=True)["SalePrice"].drop("SalePrice").sort_values(ascending=False)
    display(corr.head(15).to_frame("correlation_with_SalePrice"))


## 2. Feature engineering

In [ ]:
def add_features(df):
    out = df.copy()
    def num(c): return pd.to_numeric(out[c], errors="coerce") if c in out else None
    living, basement = num("LivAreaSF"), num("TotalBsmtSF")
    if basement is None:
        b1, b2 = num("BsmtFinSF1"), num("BsmtFinSF2")
        if b1 is not None or b2 is not None: basement = (b1 if b1 is not None else 0) + (b2 if b2 is not None else 0)
    if living is not None and basement is not None: out["TotalLivingArea"] = living.fillna(0) + basement.fillna(0)
    porch_cols = [num(c) for c in ["OpenPorchSF","EnclosedPorch","3SsnPorch","ScreenPorch","WoodDeckSF"] if c in out]
    if porch_cols: out["TotalPorchSF"] = pd.concat(porch_cols, axis=1).fillna(0).sum(axis=1)
    quality = num("OverallQual")
    if quality is not None and living is not None: out["Quality_LivingArea"] = quality * living
    sold, built, remodel = num("YrSold"), num("YearBuilt"), num("YearRemodAdd")
    if sold is not None and built is not None: out["HouseAgeAtSale"] = (sold-built).clip(lower=0)
    if sold is not None and remodel is not None: out["YearsSinceRemodel"] = (sold-remodel).clip(lower=0)
    weights = {"FullBath":1.0,"HalfBath":0.5,"BsmtFullBath":1.0,"BsmtHalfBath":0.5}
    baths = [(num(c),w) for c,w in weights.items() if c in out]
    if baths: out["TotalBathrooms"] = sum(s.fillna(0)*w for s,w in baths)
    rooms = num("TotRmsAbvGrd")
    if rooms is not None and living is not None: out["RoomsPer1000SqFt"] = rooms/(living.replace(0,np.nan)/1000)
    for c in ["LotArea","LivAreaSF","TotalBsmtSF","1stFlrSF","2ndFlrSF","GarageArea"]:
        v = num(c)
        if v is not None: out["Log1p_"+c] = np.log1p(v.clip(lower=0))
    for c in list(out.columns):
        if out[c].isna().any(): out[c+"_was_missing"] = out[c].isna().astype("int8")
    return out


## 3. Prepare data and split validation rows

In [ ]:
y = train["SalePrice"].astype(float)
X = add_features(train.drop(columns=["SalePrice"]))
X_test = add_features(test.copy())
X, X_test = X.align(X_test, join="outer", axis=1)
cat_cols = X.select_dtypes(include=["object","category","string"]).columns.tolist()
for c in cat_cols:
    X[c] = X[c].fillna("__MISSING__").astype(str)
    X_test[c] = X_test[c].fillna("__MISSING__").astype(str)
X, X_test = X.replace([np.inf,-np.inf],np.nan), X_test.replace([np.inf,-np.inf],np.nan)
cat_idx = [X.columns.get_loc(c) for c in cat_cols]
Xtr, Xv, ytr, yv = train_test_split(X,y,test_size=0.2,random_state=42)
print(Xtr.shape, Xv.shape, "categorical features:", len(cat_idx))


## 4. Model comparison: ExtraTrees baseline vs CatBoost MAE

In [ ]:
# ExtraTrees baseline with one-hot encoded categorical features.
A, B = pd.get_dummies(Xtr, dummy_na=True), pd.get_dummies(Xv, dummy_na=True)
A, B = A.align(B, join="left", axis=1, fill_value=0)
med = A.median(numeric_only=True)
A, B = A.fillna(med).fillna(0), B.fillna(med).fillna(0)
extra = ExtraTreesRegressor(n_estimators=500, random_state=42, n_jobs=-1)
extra.fit(A,ytr); p_extra = np.maximum(extra.predict(B),1)
mape_extra = mean_absolute_percentage_error(yv,p_extra)*100
cat = CatBoostRegressor(loss_function="MAE", eval_metric="MAPE", iterations=2500,
    depth=5, learning_rate=0.025, l2_leaf_reg=10, random_seed=42,
    verbose=250, allow_writing_files=False)
cat.fit(Xtr,ytr,eval_set=(Xv,yv),cat_features=cat_idx,early_stopping_rounds=200,verbose=250)
p_cat = np.maximum(cat.predict(Xv),1)
mape_cat = mean_absolute_percentage_error(yv,p_cat)*100
print(f"ExtraTrees hold-out MAPE: {mape_extra:.4f}%")
print(f"CatBoost hold-out MAPE: {mape_cat:.4f}%")
print(f"CatBoost hold-out MAE: {mean_absolute_error(yv,p_cat):.2f}")
display(pd.DataFrame({"Model":["ExtraTrees baseline","CatBoost MAE"],"Validation MAPE (%)":[mape_extra,mape_cat]}).sort_values("Validation MAPE (%)"))


## 5. Train final five-seed ensemble and save submission

In [ ]:
# Five-seed ensemble matching the best reported model family and parameters.
# For exact replication, use the median best-iteration count saved from your original Day 22 run.
SEEDS = [42,123,2026,7,21]
FINAL_ITERATIONS = 2500
preds = []
for seed in SEEDS:
    model = CatBoostRegressor(loss_function="MAE", iterations=FINAL_ITERATIONS,
        depth=5, learning_rate=0.025, l2_leaf_reg=10, random_seed=seed,
        verbose=250, allow_writing_files=False)
    model.fit(X,y,cat_features=cat_idx,verbose=250)
    preds.append(model.predict(X_test))
submission = pd.DataFrame({"Id":test["Id"],"SalePrice":np.maximum(np.mean(preds,axis=0),1)})
out = Path("/kaggle/working/submission.csv")
submission.to_csv(out,index=False)
assert list(submission.columns) == ["Id","SalePrice"]
assert len(submission) == len(test)
assert submission["SalePrice"].notna().all() and (submission["SalePrice"] > 0).all()
display(submission.head())
print("Saved", out, "| rows:", len(submission), "| missing predictions:", submission["SalePrice"].isna().sum())


## 6. Final result and notes

Best score reported during this project: **9.11091** on the public leaderboard with the five-seed CatBoost MAE ensemble. The public leaderboard score may differ from the final/private leaderboard score.

| Experiment | Model | Public score |
|---|---|---:|
| Day 18 | CatBoost | 9.49087 |
| Day 21 | Feature engineering + CatBoost MAE | 9.46597 |
| Day 22 | Five-seed CatBoost MAE ensemble | **9.11091** |

Download `submission.csv` from the notebook Output panel and upload it via the competition's **Submit Predictions** page. Save a notebook version after successful execution.
